# NB5 · Safety guardrails and compliance

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What this notebook does

This notebook does not improve the performance of the model. It determines when the model
should not speak.

Three guardrails are added. The model stays silent when it is undecided. It produces no
prediction for a patient unlike the training population. It proposes no decision for a
patient about whom too little is known.

At the end a compliance report is produced. It is printed to the screen and records what
your system is, what it does not do, and which questions remain open.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Code carried from the previous notebook

Paste the whole block collected at the end of the previous notebook into the cell
below. Do not delete the `#@cdss` marker on the first line; that block is collected
again at the end of this notebook and carried to the next one.

Running the block rebuilds everything you wrote in the earlier notebooks. Where it
reads data from the web, the cell may take a few seconds.


In [ ]:
#@cdss onceki_defter
# Paste the generated code below this line.


### Check · The carried code


In [ ]:
kit.check_defined('model', 'X_test', 'y_test', 'probability', 'result', 'reasoning_table')


---

## Step 1 · Guarded prediction

Until now the system produced a probability for every patient. From here it will decline
in three situations.

**Indecision.** Where the probability sits very close to the threshold the model is
undecided. Leaving the decision to the clinician is more honest than producing a
prediction.

**An unfamiliar patient.** When a patient unlike anyone in the training data arrives, the
model still returns a number, but that number rests on nothing.

**Missing information.** For a patient most of whose information is empty, the filling
step closes every gap with average values and the model returns a confident number. The
system ends up proposing a decision for a patient about whom nothing is known. No error
is raised.

Where the guardrail thresholds sit is a clinical decision. Those are the numbers you write
into the prompt, and you are expected to note your reasons.


### Prompt 1

```
Write a piece of work that adds three safety guardrails to the system. Name it
guarded_prediction and let it take one patient's information.

Have it check the following in order:
1. If more than 60 percent of the patient's information is empty, produce no prediction.
   Leave the decision to the clinician on grounds of insufficient information.
2. If the patient is unlike anyone in the training group, produce no prediction. Measure
   unlikeness by how far the patient sits from the typical values of the training group.
3. If both of these pass, compute the probability. If the probability sits very close to
   the decision threshold, that is within 0.05 of 0.50, report that the model is
   undecided.
4. If none of these apply, give the normal decision.

Give the result back as a dictionary with a decision key holding the decision as text, a
probability key holding a number or None, and a reason key holding a short explanation.

Define the guardrail thresholds outside the piece of work, with names in capitals, and
note beside them in a comment that these are clinical decisions.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named guarded_prediction that returns a
dictionary containing decision, probability and reason.
Given a patient whose information is entirely empty, probability must be None.
```


In [ ]:
#@cdss bariyerler
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_function('guarded_prediction')


In [ ]:
import numpy as _np

# An ordinary patient
print('Ordinary patient :', guarded_prediction(X_test[0:1]))

# A patient whose information is entirely empty
empty = _np.full((1, _np.shape(X_test)[1]), _np.nan)
print('Empty patient    :', guarded_prediction(empty))

# A patient at the extremes
extreme = _np.asarray(X_test[0:1], dtype=float) * 50
print('Extreme values   :', guarded_prediction(extreme))


### Python note · Condition chains and early return

In the code you will see successive `if` lines each containing a `return`. This is called
an **early return**: when the condition holds, the piece of work gives its result
immediately and never looks at the lines below.

The structure is preferred in clinical safety code because the order carries meaning. The
most serious obstacle is checked first, then the next. Asking whether a patient resembles
the training population is meaningless when their information is insufficient in the first
place.

That is also why the thresholds are defined outside the piece of work: numbers hidden in
the middle of the code cannot be audited. In a clinical system the threshold in use is
information that has to be documented.


---

## Step 2 · Red team

You will push the system with scenarios of your own. This is called a red team exercise.

At least one of the five inputs must not be obviously broken but a clinically plausible,
unusual patient. That row is the important one; a system that fails only on obviously
broken input has not really been tested.


### Prompt 2

```
Construct five different patient examples that push the system and show what
guarded_prediction does with each.

Let the five be:
1. An ordinary patient, nothing changed.
2. A patient one of whose values is physiologically impossible.
3. A patient many of whose values sit far from the typical values of the training group.
4. A patient almost all of whose information is empty.
5. A clinically plausible patient with no obvious defect, but one of whose values sits
   near the extremes of the training group.

Give the results back as a table with the columns example, decision, probability and
reason. Keep it under the name red_team and show it.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named red_team with five rows containing example, decision,
probability and reason.
```


In [ ]:
#@cdss kirmizi_takim
# Paste the generated code below this line.


### Check 2


In [ ]:
kit.check_frame(red_team, name='red_team',
                required=['example', 'decision', 'probability', 'reason'], min_rows=5)
print()
print(red_team.to_string(index=False))


### Reading the table

Look at the fifth row. What did the system do with a clinically plausible but unusual
patient? If it produced a prediction, can that prediction be trusted?

The fourth row matters too. Before the guardrail was added the system returned a confident
probability for that patient. You added the guardrail; without it no error would have been
raised.


---

## Step 3 · The compliance report

In the final step you will produce a report recording what your system is. It is printed
to the screen and participants can take it back to their own institutions.

Its content comes from the framework set out in the lecture on 16 September: the purpose
of the system, its data, the performance measured, its guardrails, its known limitations
and its regulatory position.

The regulatory section needs care. The compliance dates in this area changed in July
2026; in the European Union the operative date for artificial intelligence inside medical
devices was deferred to 2 August 2028. The change is recent enough that many AI tools
still return the superseded timetable. Check the date the tool gives you; it is a concrete
instance of confident invention.


### Prompt 3

```
Write a piece of work that produces a compliance report for my system. Name it
compliance_report and have it give the report back as a single piece of text.

The report must contain these headings, filled in from the information I have:
  PURPOSE       -> which clinical decision the system supports, when it intervenes
  USER          -> who will use it
  DATA          -> how many patients, how many records, which source, single centre
  PERFORMANCE   -> the measurements in the result dictionary and what they mean
  GUARDRAILS    -> which guardrails exist, their thresholds, who set them
  LIMITATIONS   -> what the system does not do, which patient groups it does not cover
  REGULATORY    -> whether this software counts as a medical device, which legislation
                   applies, what the operative compliance date is
  OPEN_QUESTIONS -> points a lawyer and a clinician would need to check

In the PERFORMANCE section take the numbers from the result dictionary; do not type them
by hand.

In the REGULATORY section state your confidence beside every claim. Present none of it as
legal advice.

Then produce the report, keep it under the name report and print it.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named compliance_report that returns text.
There must be text named report containing all eight headings above.
```


In [ ]:
#@cdss uyumluluk
# Paste the generated code below this line.


### Check 3


In [ ]:
kit.check_report(report, name='report',
                 must_contain=['PURPOSE', 'USER', 'DATA', 'PERFORMANCE', 'GUARDRAILS',
                               'LIMITATIONS', 'REGULATORY', 'OPEN_QUESTIONS'])


In [ ]:
print(report)


### Auditing the report

Look at three things.

**The date in the REGULATORY section.** In the European Union the operative date for
artificial intelligence inside medical devices is 2 August 2028. If the tool gave 2027 or
earlier, it returned the superseded timetable. Correct it and note it; expect the same
behaviour on other regulatory questions.

**The numbers in the PERFORMANCE section.** Are they the same as the values in the
`result` dictionary? If the tool wrote a figure of its own invention, the report cannot be
trusted.

**Any citations.** Ask for a DOI or official document number for every paper or guideline
cited. Where none can be given, the citation should be removed.


---

## End of notebook · Collecting the code

The cell below collects the code you carried from the earlier notebooks together with
what you added here, as one block. Copy the whole block; you will paste it into the
first cell of NB6.

The block is also saved as `cdss_nb5.py`. That file disappears when the Colab session closes,
so keep a copy in a text file on your own computer as well.


In [ ]:
code_so_far = kit.export(save_as='cdss_nb5.py')


## What this notebook did

Three guardrails, a red team exercise and a compliance report were added to the system.

The guardrail thresholds are clinical decisions you made, and they are written into the
report. The threshold a system runs at is information as important as how well it runs.

---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The dataset
you used is an open collection prepared for teaching and does not represent the patient
population of your own institution. The material is for teaching.
